# Method A with Restricted Label Space (Insect Orders Only)

Tests BioCLIP's zero-shot performance when the label space is restricted to species within the 14 target insect orders, rather than all species in BioCLIP's pretraining vocabulary.

**Purpose:** isolates whether restricting label space at the species level — keeping discriminative species strings but removing irrelevant orders — recovers performance, or whether the fundamental issue lies in BioCLIP's handling of short, prefix-sharing order-level strings.

**Comparison context:**
- **Method A standard** (Act 1): all species in BioCLIP's vocabulary → weighted F1 ≈ 0.861
- **Method A restricted** (this experiment): only species in 14 target orders → expected better
- **Method B** (Act 1): direct 14 short order strings → weighted F1 ≈ 0.014
- **Soft-prompts trained** (Act 2): learned discriminative tokens at prefix → weighted F1 ≈ 0.971

If Method A restricted substantially improves over Method A standard but doesn't reach soft-prompt levels, it confirms that label space matters but discriminative text representations remain the bottleneck for closed-set order classification.

**Outputs:**
- `output_method_a_restricted/per_class_results.csv` — per-class precision/recall/F1
- `output_method_a_restricted/summary.csv` — aggregate metrics

## Imports and configuration

In [ ]:
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
import open_clip

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
CONFIG = {
    "model_name": "hf-hub:imageomics/bioclip",
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "meta_path": "/workspace/thesis/ultimate_bioclip_top1_rank_order_meta_12062025.csv",
    "image_root": "/workspace/thesis/ALL_copy",
    "output_dir": "/workspace/thesis/output_bioClip_restricted_labels",

    "image_id_col": "image_id",
    "label_col": "label",

    "batch_size": 64,
    "num_workers": 4,

    # 14 target orders (present in your data)
    "target_orders": [
        "Araneae", "Blattodea", "Coleoptera", "Diptera", "Hemiptera",
        "Hymenoptera", "Ixodida", "Lepidoptera", "Mecoptera", "Neuroptera",
        "Orthoptera", "Plecoptera", "Raphidioptera", "Trichoptera",
    ],

    # Distractor orders: realistic taxonomic alternatives that BioCLIP could pick.
    # Including these gives the model "permission to disagree" with the 14-class
    # assumption, providing a more honest evaluation than label space restricted
    # to only target orders.
    "distractor_orders": [
        # Other insect orders (Insecta)
        "Odonata", "Ephemeroptera", "Mantodea", "Phasmida",
        "Dermaptera", "Psocodea", "Thysanoptera", "Embioptera",
        "Strepsiptera", "Megaloptera", "Zygentoma", "Archaeognatha",
        "Siphonaptera",
        # Other Arachnida orders
        "Opiliones", "Scorpiones", "Pseudoscorpiones", "Solifugae",
        "Amblypygi", "Mesostigmata", "Trombidiformes", "Sarcoptiformes",
        # Other Arthropoda — small ones occasionally found in pan traps
        "Isopoda", "Amphipoda",
    ],
}

# Combine target + distractors for the actual prediction label space
CONFIG["full_label_space"] = CONFIG["target_orders"] + CONFIG["distractor_orders"]

CROP_ID_PATTERN = re.compile(r"(\d{14}_crop_[a-zA-Z0-9]+\.jpg)$")
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

print(f"Target orders ({len(CONFIG['target_orders'])}): {CONFIG['target_orders']}")
print(f"Distractor orders ({len(CONFIG['distractor_orders'])}): {CONFIG['distractor_orders']}")
print(f"Full label space size: {len(CONFIG['full_label_space'])}")
print(f"Output: {CONFIG['output_dir']}")

## Building candidate species list

For "Method A restricted" ((BioClip2 with restricted label space), we need a list of species strings within the 14 target orders. We use BioCLIP 2's pretraining vocabulary (TreeOfLife taxonomic tree) and filter to keep only species whose order is in our target set.

The simplest source: pybioclip's bundled taxonomic data. If pybioclip is not installed, we can construct a representative species list from the model's known vocabulary by querying it programmatically.

For this experiment, we use the pybioclip library which has the taxonomic vocabulary built-in.

In [ ]:
# Installing pybioclip if not present
import subprocess
import sys

try:
    import pybioclip
    print("pybioclip already installed")
except ImportError:
    print("Installing pybioclip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pybioclip"])
    import pybioclip
    print("Done.")

print(f"pybioclip version: {pybioclip.__version__}")

In [ ]:
from bioclip import TreeOfLifeClassifier, Rank

# Loading BioCLIP classifier
classifier = TreeOfLifeClassifier(
    model_str=CONFIG["model_name"],
    device=CONFIG["device"],
)
print(f"Classifier loaded: {CONFIG['model_name']}")

## Build path index and load metadata

In [ ]:
def build_path_index(image_root):
    image_root = Path(image_root)
    files = list(image_root.glob("*.jpg"))
    path_index = {}
    for f in files:
        m = CROP_ID_PATTERN.search(f.name)
        if m:
            path_index[m.group(1)] = str(f)
    print(f"Mapped {len(path_index)} image_ids")
    return path_index


path_index = build_path_index(CONFIG["image_root"])

meta = pd.read_csv(CONFIG["meta_path"])
meta = meta.dropna(subset=[CONFIG["label_col"], CONFIG["image_id_col"]]).reset_index(drop=True)
meta["local_path"] = meta[CONFIG["image_id_col"]].map(path_index)
matched = meta.dropna(subset=["local_path"]).reset_index(drop=True)

y_str = matched[CONFIG["label_col"]].to_numpy()
image_paths = matched["local_path"].to_numpy()
classnames = sorted(np.unique(y_str).tolist())

print(f"Matched: {len(matched)} crops")
print(f"Classes ({len(classnames)}): {classnames}")

## Running Method A restricted: classify with order rank, restricted to target orders

pybioclip's `TreeOfLifeClassifier.predict()` with `rank=Rank.ORDER` aggregates over species within each order. By setting `class_str` to our 14 target orders, the classifier compares image features against the species under those orders only (not all species in BioCLIP's vocabulary).

This is the **restricted Method A**: same species-level comparison as standard Method A, but the label space is filtered to species within our 14 target orders.

In [ ]:
print(f"Running Method A restricted on {len(matched)} images...")
print(f"Label space: {len(CONFIG['target_orders'])} target orders + "
      f"{len(CONFIG['distractor_orders'])} distractor orders = "
      f"{len(CONFIG['full_label_space'])} total orders\n")

start = time.time()
predictions = []
predicted_distractor_count = 0  # tracking how often model picks a distractor

batch_size = CONFIG["batch_size"]
n_batches = (len(image_paths) + batch_size - 1) // batch_size

for batch_idx in range(n_batches):
    batch_start = batch_idx * batch_size
    batch_end = min(batch_start + batch_size, len(image_paths))
    batch_paths = image_paths[batch_start:batch_end].tolist()

    # Predicting order, restricted to target + distractor orders
    results = classifier.predict(
        images=batch_paths,
        rank=Rank.ORDER,
        k=1,  # top-1 prediction
        class_str=CONFIG["full_label_space"],  # 14 targets + distractors
    )

    # Extracting predicted order for each image
    for image_path in batch_paths:
        image_results = [r for r in results if r["file_name"] == image_path]
        if image_results:
            predicted_order = image_results[0]["order"]
            if predicted_order in CONFIG["distractor_orders"]:
                predicted_distractor_count += 1
        else:
            predicted_order = None  # should not happen with k=1
        predictions.append(predicted_order)

    if batch_idx % 50 == 0:
        elapsed = time.time() - start
        eta = elapsed / (batch_idx + 1) * (n_batches - batch_idx - 1)
        print(f"  Batch {batch_idx+1}/{n_batches}: "
              f"elapsed={timedelta(seconds=int(elapsed))}, "
              f"ETA={timedelta(seconds=int(eta))}")

total_time = time.time() - start
print(f"\nTotal inference time: {timedelta(seconds=int(total_time))}")
print(f"Predictions made: {len(predictions)}")
print(f"Predicted as distractor (not in target 14): {predicted_distractor_count} "
      f"({100*predicted_distractor_count/len(predictions):.2f}%)")

## Evaluate predictions

In [ ]:
# Converting predictions to numpy and compute report
y_true = y_str
y_pred = np.array(predictions)

# Handling any None predictions (shouldn't happen but just in case)
valid_mask = y_pred != None
print(f"Valid predictions: {valid_mask.sum()}/{len(y_pred)}")
y_true_valid = y_true[valid_mask]
y_pred_valid = y_pred[valid_mask]

# Classification report
report = classification_report(
    y_true_valid, y_pred_valid,
    labels=classnames, target_names=classnames,
    output_dict=True, zero_division=0,
)

# Per-class results
per_class_rows = []
for cls_name in classnames:
    stats = report.get(cls_name, {})
    per_class_rows.append({
        "class": cls_name,
        "precision": round(stats.get("precision", 0.0), 4),
        "recall": round(stats.get("recall", 0.0), 4),
        "f1": round(stats.get("f1-score", 0.0), 4),
        "support": int(stats.get("support", 0)),
    })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(Path(CONFIG["output_dir"]) / "per_class_results.csv", index=False)
print("\n=== Per-class results ===")
print(per_class_df.to_string(index=False))

In [ ]:
# Aggregate metrics
weighted_f1 = report["weighted avg"]["f1-score"]
macro_f1_all = report["macro avg"]["f1-score"]

# macro F1 restricted to n>=50 classes
eligible_classes = [c for c in classnames if (y_str == c).sum() >= 50]
eligible_f1s = [report[c]["f1-score"] for c in eligible_classes if c in report]
macro_f1_restricted = float(np.mean(eligible_f1s)) if eligible_f1s else 0.0

hemiptera_f1 = report.get("Hemiptera", {}).get("f1-score", 0.0)
coleoptera_f1 = report.get("Coleoptera", {}).get("f1-score", 0.0)

summary_df = pd.DataFrame([{
    "method": "method_a_restricted",
    "weighted_f1": round(weighted_f1, 4),
    "macro_f1_all": round(macro_f1_all, 4),
    "macro_f1_restricted_n50": round(macro_f1_restricted, 4),
    "Hemiptera_f1": round(hemiptera_f1, 4),
    "Coleoptera_f1": round(coleoptera_f1, 4),
    "inference_time_sec": int(total_time),
}])
summary_df.to_csv(Path(CONFIG["output_dir"]) / "summary.csv", index=False)
print("\n=== Summary ===")
print(summary_df.to_string(index=False))

## Output files summary

| File | Purpose |
|---|---|
| `per_class_results.csv` | Per-class precision/recall/F1 |
| `summary.csv` | Aggregate metrics (weighted F1, macro F1, Hemiptera F1, Coleoptera F1) |